# Chapter 6 — Potential Sweep Methods: Non-Reversible Reactions

*Python-native adaptation of Michael Honeychurch, **Simulating Electrochemical
Reactions in Mathematica** (SERM), Chapter 6, "Potential sweep methods —
non-reversible reactions," together with the companion simulator notebook
`Extra Notebooks/chapter6/ImplicitCVQuasi.nb`.*

Chapter 5 treated the **reversible** cyclic voltammogram: the electrode was
assumed to be at Nernstian equilibrium at every instant, so the surface
concentration ratio $c_O/c_R$ followed the applied potential exactly. That is
the fast-kinetics idealisation. Real electron transfer proceeds at a finite
rate set by the heterogeneous standard rate constant $k_s$ and the transfer
coefficient $\alpha$, described by the **Butler–Volmer** equation. When $k_s$ is
not large compared with the rate at which the potential is swept, the electrode
cannot keep up: the wave broadens, the peak current drops, and the peak
potential walks away from $E^0$. This chapter builds a single implicit
finite-difference solver that spans the whole range — from reversible, through
**quasi-reversible**, to **totally irreversible** — and shows how the kinetics
distort the wave.

What this chapter delivers:

* the dimensionless Butler–Volmer surface boundary condition and how it reduces
  to the Chapter 5 Nernstian condition in the fast limit;
* a vectorised, fully implicit FD simulator (`serm.ch06_*`) reusing the project
  machinery;
* figures of the wave across the reversibility spectrum and of the
  Nicholson–Shain peak-shift / peak-current trends;
* a **validation** section that pins three independent analytic anchors — the
  Cottrell grid check, the reversible Nicholson–Shain peak $0.4463$, and the
  totally irreversible peak coefficient $0.4958\sqrt{\alpha}$ — with `assert`s.


## Theory — from Butler–Volmer to a dimensionless boundary condition

For the reduction $O + n e^- \rightleftharpoons R$ the net faradaic flux at the
electrode is given by the **Butler–Volmer** rate law,

$$
\frac{i}{nFA} \;=\; k_s\Bigl[\,c_O(0,t)\,e^{-\alpha\, f (E-E^0)}
\;-\; c_R(0,t)\,e^{(1-\alpha)\, f (E-E^0)}\Bigr],
\qquad f \equiv \frac{nF}{RT},
$$

where $c_O(0,t)$, $c_R(0,t)$ are the **surface** concentrations (Bard & Faulkner,
2nd ed., eq. 3.3.11). The reversible Nernstian condition of Chapter 5 is the
limit $k_s\to\infty$ of this law: a finite flux then forces the bracket to zero,
i.e. $c_O/c_R = e^{f(E-E^0)} \equiv \xi$.

### Non-dimensionalisation

Following SERM Ch. 5–6 we scale the problem by the sweep itself. The
dimensionless potential

$$
p \;=\; f\,(E - E^0)
$$

advances linearly in time during a ramp, so $p$ doubles as a dimensionless time
$z = \sigma t$ with $\sigma = f v$ (the dimensionless sweep rate
`serm.waveforms.dimensionless_sweep_rate`). Distance is scaled as
$X = x\sqrt{\sigma/D}$, and concentrations by the bulk value $c^*$. Dividing the
Butler–Volmer law by $nFA c^*\sqrt{\sigma D}$ gives the **dimensionless rate
constant**

$$
k_s^{\dim} \;=\; \frac{k_s}{\sqrt{f\,v\,D}},
$$

the natural reversibility yardstick: $k_s^{\dim}\gg 1$ is reversible,
$k_s^{\dim}\ll 1$ irreversible. Note its $v^{-1/2}$ dependence — *raising the
scan rate makes any couple look less reversible*, which is the experimental knob
behind the Nicholson–Shain diagnostics.

## Theory — implicit finite differences with a kinetic surface

The interior obeys the dimensionless diffusion equation
$\partial c_O/\partial z = \partial^2 c_O/\partial X^2$. Discretising fully
implicitly on a uniform grid $X_j = (j-1)\,\Delta X$, $z_k = k\,\Delta\tau$ and
writing the **model diffusion number** $D_M = \Delta\tau/\Delta X^2$, every
interior node gives the familiar tridiagonal row

$$
-D_M\,c_{j-1}^{k+1} + (1+2D_M)\,c_j^{k+1} - D_M\,c_{j+1}^{k+1} \;=\; c_j^{k}.
$$

The difficulty is the **surface node** $j=1$, where the unknown
$c_{O,1}^{k+1}$ also appears in the Butler–Volmer flux. SERM Ch. 6 resolves it
by writing the surface flux with a three-point one-sided derivative and using
$c_R = 1 - c_O$ (equal diffusion coefficients), then *solving algebraically* for
$c_{O,1}^{k+1}$:

$$
\boxed{\;c_{O,1}^{k+1} = \bigl(k_s^{*}\,\xi^{1-\alpha} + 4\,c_{2}^{k+1} - c_{3}^{k+1}\bigr)\,
\mathrm{tmp},\qquad
\mathrm{tmp} = \frac{\xi^{\alpha}}{3\,\xi^{\alpha} + k_s^{*}\,(1+\xi)}\;}
$$

with $\xi = e^{p}$ and the *grid-scaled* rate constant
$k_s^{*} = 2\,k_s^{\dim}\sqrt{z_{\text{span}} / (D_M (n-1))}$. Substituting this
boundary value back into the first tridiagonal row only modifies the surface
diagonal and super-diagonal each step:

$$
Y_2 \leftarrow (1+2D_M) - 4 D_M\,\mathrm{tmp},\qquad
Z_2 \leftarrow -D_M + D_M\,\mathrm{tmp},
$$

and adds $\mathrm{tmp}\,D_M\,k_s^{*}\,\xi^{1-\alpha}$ to the right-hand side. As
$k_s^{*}\to\infty$, $\mathrm{tmp}\to 0$ and $c_{O,1}\to \xi/(1+\xi)$: the
Nernstian Dirichlet boundary of Chapter 5 is recovered exactly. This is the
single property that lets one solver cover the whole reversibility range.

The dimensionless current reported is the surface gradient
$\chi_{\mathrm{grad}} = \partial c_O/\partial X|_{X=0}$, evaluated with the same
three-point stencil. It maps to the laboratory current by
$i = nFA c^*\sqrt{D\sigma}\,\chi_{\mathrm{grad}}$; for a reduction the forward
(cathodic) branch is negative.

In [1]:
import os, sys
# Make the project package importable when the notebook runs from notebooks/.
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import matplotlib.pyplot as plt

import serm
from serm import ch06_potential_sweep_nonreversible as ch6
from serm.waveforms import dimensionless_sweep_rate
import serm.echem as echem

np.set_printoptions(precision=4, suppress=True)

## The simulator

`serm.ch06_potential_sweep_nonreversible.simulate_cv` implements the scheme
above. It sweeps the dimensionless potential triangularly from `upper` down to
`lower` and back, patches the surface row each step, and solves the tridiagonal
system with `scipy.linalg.solve_banded`. A `CVResult` carries the potential axis
`p`, the current `chi`, and convenience properties for the forward (cathodic)
peak.

The default scan window `upper = 11.6435`, `lower = -15.5766` (in units of
$f(E-E^0)$) is taken from SERM's `ImplicitCVQuasi.nb`; it is wide enough to start
and end on the diffusion-limited baselines. We first confirm the dimensionless
rate constant grouping against `serm`'s sweep-rate helper.

In [2]:
# A representative couple: D = 1e-5 cm^2/s, ks = 1e-3 cm/s, v = 1 V/s.
F, R, T = 96485.33212, 8.314462618, 298.15
f = F / (R * T)
D, ks, v, alpha = 1e-5, 1e-3, 1.0, 0.5

ks_dim = ch6.ks_dimensionless(ks, v, D, T)
sigma = dimensionless_sweep_rate(v, n_electrons=1, temperature=T)
print(f"f = nF/RT            = {f:.4f} 1/V")
print(f"sigma = f v          = {sigma:.4f} 1/s   (serm.waveforms)")
print(f"ks_dim = ks/sqrt(fvD)= {ks_dim:.5f}      (reversibility yardstick)")

f = nF/RT            = 38.9217 1/V
sigma = f v          = 38.9217 1/s   (serm.waveforms)
ks_dim = ks/sqrt(fvD)= 0.05069      (reversibility yardstick)


### A voltammogram across the reversibility spectrum

Holding everything else fixed and varying only $k_s^{\dim}$ traces the wave from
reversible to irreversible. As the kinetics slow, the cathodic peak shifts to
more negative potential, the wave broadens, and the forward/reverse peaks pull
apart — the qualitative signature of sluggish electron transfer.

In [3]:
ks_dims = [1e3, 1.0, 1e-1, 1e-2]
labels  = ["reversible (ks_dim=1e3)", "ks_dim=1", "ks_dim=0.1", "ks_dim=0.01"]
results = [ch6.simulate_cv(kd, alpha=alpha, dz=0.05, lower=-30.0) for kd in ks_dims]

fig, ax = plt.subplots(figsize=(6.4, 4.6))
for r, lab in zip(results, labels):
    # plot -chi so cathodic (reduction) peaks point up, matching SERM's sign.
    ax.plot(r.p, -r.chi, lw=1.3, label=lab)
ax.axhline(0.0, color="0.7", lw=0.6)
ax.set_xlabel(r"$p = (nF/RT)(E-E^0)$")
ax.set_ylabel(r"$-\chi_\mathrm{grad}$  (cathodic current up)")
ax.set_title("Cyclic voltammograms vs. electron-transfer reversibility")
ax.invert_xaxis()  # conventional CV: potential decreases left-to-right on reduction
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

for r, lab in zip(results, labels):
    print(f"{lab:28s} cathodic peak = {r.cathodic_peak:.4f}  at p = {r.cathodic_peak_potential:+.2f}")

reversible (ks_dim=1e3)      cathodic peak = 0.4462  at p = -1.11
ks_dim=1                     cathodic peak = 0.4066  at p = -2.01
ks_dim=0.1                   cathodic peak = 0.3555  at p = -5.51
ks_dim=0.01                  cathodic peak = 0.3501  at p = -10.12


/tmp/ipykernel_1423078/827603883.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Nicholson–Shain trends: peak shift and peak current

The diagnostic value of these curves is quantitative. Nicholson & Shain (1964)
showed that for a **totally irreversible** wave the peak potential moves by

$$
\Delta E_p \;=\; -\frac{2.303\,RT}{2\,\alpha n F}\;\text{per decade of }v
\;\approx\; -\frac{29.6}{\alpha n}\ \text{mV/decade}\ \text{(298 K)},
$$

and the peak current keeps the diffusion $\sqrt{v}$ scaling but with the
coefficient $0.4958\sqrt{\alpha}$ instead of the reversible $0.4463$ (Bard &
Faulkner, 2nd ed., eqs. 6.3.1 and 6.3.8). Because $k_s^{\dim}\propto v^{-1/2}$, a
*decade in scan rate is half a decade in* $k_s^{\dim}$; equivalently the peak
shifts by $-\ln 10/\alpha = -4.605$ in dimensionless $p$ per decade of
$k_s^{\dim}$. We sweep $k_s^{\dim}$ over four decades and read off both trends.

In [4]:
kd_scan = np.logspace(1.0, -4.0, 16)
peaks_chi, peaks_p = [], []
for kd in kd_scan:
    r = ch6.simulate_cv(kd, alpha=alpha, dz=0.05, lower=-65.0)
    peaks_chi.append(r.cathodic_peak)
    peaks_p.append(r.cathodic_peak_potential)
peaks_chi = np.array(peaks_chi)
peaks_p = np.array(peaks_p)

fig, (axA, axB) = plt.subplots(1, 2, figsize=(10.5, 4.2))

axA.semilogx(kd_scan, peaks_chi, "o-", lw=1.2, ms=4)
axA.axhline(0.4463, color="C1", ls="--", lw=1.0, label="reversible 0.4463")
axA.axhline(0.4958*np.sqrt(alpha), color="C3", ls=":", lw=1.2,
            label=r"irreversible $0.4958\sqrt{\alpha}$")
axA.set_xlabel(r"$k_s^{\dim}$")
axA.set_ylabel(r"cathodic peak $-\chi_\mathrm{grad}$")
axA.set_title(r"Peak current: reversible $\to$ irreversible plateau")
axA.legend(frameon=False, fontsize=8)

axB.semilogx(kd_scan, peaks_p, "s-", lw=1.2, ms=4, color="C2")
axB.set_xlabel(r"$k_s^{\dim}$")
axB.set_ylabel(r"peak potential $p_p$")
axB.set_title("Peak potential walks negative as kinetics slow")
fig.tight_layout()
plt.show()

/tmp/ipykernel_1423078/1888429652.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Quantify the irreversible peak-potential slope (deep-irreversible decades).
mask = kd_scan <= 1e-1
slope = np.polyfit(np.log10(kd_scan[mask]), peaks_p[mask], 1)[0]
print(f"dp_peak / d(log10 ks_dim) measured = {slope:+.4f}")
print(f"theory +ln(10)/alpha               = {np.log(10)/alpha:+.4f}")
# Convert to the experimental mV/decade-of-v form. Because ks_dim ~ v^(-1/2),
# a decade of v is half a decade of ks_dim, so dEp/dlog10(v) = -0.5 * slope / f.
mV_per_decade_v = -slope * 0.5 / f * 1e3
bf_ref = -2.303 * R * T / (2 * alpha * F) * 1e3   # = -29.6/alpha mV at 298 K
print(f"=> {mV_per_decade_v:+.1f} mV per decade of scan rate "
      f"(B&F 2.303RT/(2 alpha nF) = {bf_ref:+.1f})")

dp_peak / d(log10 ks_dim) measured = +4.6020
theory +ln(10)/alpha               = +4.6052
=> -59.1 mV per decade of scan rate (B&F 2.303RT/(2 alpha nF) = -59.2)


## Validation

We use the **preferred** strategy of the authoring spec: independent closed-form
/ limiting analytic checks, none of which copies Honeychurch's printed numbers.
Three anchors certify the solver:

1. **Grid certification (Cottrell).** A potential step to a fully depleted
   surface has the exact dimensionless flux $\partial c/\partial X|_0 =
   1/\sqrt{\pi\tau}$. Reproducing it confirms the spatial stencil, the
   $\Delta X = \sqrt{\Delta\tau/D_M}$ scaling, and the gradient operator —
   independently of any voltammetry.
2. **Reversible limit (Nicholson–Shain).** As $k_s^{\dim}\to\infty$ the
   Butler–Volmer wave must collapse onto the Chapter 5 Nernstian wave, whose
   cathodic peak is the textbook value $\pi^{1/2}\chi_p = 0.4463$ at
   $p \approx -1.11$. We check the kinetic solver against both this constant and
   an independent Dirichlet (Nernstian) re-simulation.
3. **Irreversible limit.** As $k_s^{\dim}\to 0$ the peak current must approach
   the totally irreversible coefficient $0.4958\sqrt{\alpha}$, and the peak
   potential must shift by $-\ln 10/\alpha$ per decade of $k_s^{\dim}$.

In [6]:
# --- (1) Cottrell grid certification --------------------------------------
DM, n_step = 4.0, 4000
z_span, tau = 10.0, 10.0 / (n_step - 1)
dX = np.sqrt(tau / DM)
m = 1 + int(np.ceil(8 * np.sqrt(DM * n_step)))
from scipy.linalg import solve_banded
conc = np.ones(m)
ab = np.zeros((3, m - 2)); ab[1] = 1 + 2*DM; ab[0, 1:] = -DM; ab[2, :-1] = -DM
flux = np.zeros(n_step)
for k in range(n_step):
    b = conc[1:m-1].copy(); b[-1] += DM          # surface held at c=0 (Dirichlet)
    interior = solve_banded((1, 1), ab, b)
    conc = np.concatenate(([0.0], interior, [1.0]))
    flux[k] = (3*conc[0] - 4*conc[1] + conc[2]) / (2*dX)
tau_axis = tau * np.arange(1, n_step + 1)
analytic = ch6.cottrell_surface_flux(tau_axis)
# Compare over a window away from the t->0 singular edge (as the pilot excludes).
w = slice(n_step // 10, n_step)
# the (3c0-4c1+c2) stencil returns -dc/dX, so compare magnitudes.
rel_cottrell = np.max(np.abs(np.abs(flux[w]) - analytic[w]) / analytic[w])
print(f"(1) Cottrell flux  max rel. error = {rel_cottrell:.2e}")
assert rel_cottrell < 5e-3, "FD grid fails the Cottrell certification"
print("    PASS: grid + surface-gradient operator certified.")

(1) Cottrell flux  max rel. error = 1.05e-03
    PASS: grid + surface-gradient operator certified.


In [7]:
# --- (2) Reversible Nicholson-Shain peak ----------------------------------
rev = ch6.simulate_cv(1e3, alpha=alpha, dz=0.02)        # kinetic solver, fast limit
ref = ch6.simulate_reversible_cv(dz=0.02)               # independent Nernstian Dirichlet
NS_REVERSIBLE = 0.4463                                   # pi^{1/2} chi_p (Nicholson-Shain)

err_kinetic = abs(rev.cathodic_peak - NS_REVERSIBLE) / NS_REVERSIBLE
err_dirich  = abs(ref.cathodic_peak - NS_REVERSIBLE) / NS_REVERSIBLE
err_match   = abs(rev.cathodic_peak - ref.cathodic_peak) / ref.cathodic_peak
print(f"(2) kinetic ks->inf peak = {rev.cathodic_peak:.4f}  (rel err vs 0.4463: {err_kinetic:.2e})")
print(f"    Dirichlet Nernst peak = {ref.cathodic_peak:.4f}  (rel err vs 0.4463: {err_dirich:.2e})")
print(f"    peak potential p_p    = {rev.cathodic_peak_potential:+.3f}  (NS ~ -1.11)")
assert err_kinetic < 5e-3, "reversible limit does not match Nicholson-Shain 0.4463"
assert err_match   < 5e-3, "kinetic solver and Nernstian Dirichlet disagree"
assert abs(rev.cathodic_peak_potential + 1.11) < 0.2, "reversible peak at wrong potential"
print("    PASS: ks_dim -> inf recovers the Chapter 5 reversible wave.")

(2) kinetic ks->inf peak = 0.4462  (rel err vs 0.4463: 1.60e-04)
    Dirichlet Nernst peak = 0.4463  (rel err vs 0.4463: 3.84e-05)
    peak potential p_p    = -1.121  (NS ~ -1.11)
    PASS: ks_dim -> inf recovers the Chapter 5 reversible wave.


In [8]:
# --- (3) Totally irreversible limit ---------------------------------------
irr = ch6.simulate_cv(1e-4, alpha=alpha, dz=0.05, lower=-60.0)
IRREV_COEFF = 0.4958 * np.sqrt(alpha)        # B&F eq. 6.3.8: 0.4958 sqrt(alpha)
err_irrev = abs(irr.cathodic_peak - IRREV_COEFF) / IRREV_COEFF
print(f"(3) ks->0 cathodic peak  = {irr.cathodic_peak:.4f}  "
      f"(analytic 0.4958*sqrt(alpha) = {IRREV_COEFF:.4f}, rel err {err_irrev:.2e})")
assert err_irrev < 1e-2, "irreversible peak does not match 0.4958 sqrt(alpha)"

# Peak-potential slope (re-using the scan computed above for the figure).
assert abs(slope - (np.log(10)/alpha)) < 0.05, "irreversible peak-shift slope wrong"
print(f"    peak-shift slope = {slope:+.4f} per decade ks_dim "
      f"(theory {np.log(10)/alpha:+.4f})")
print("    PASS: ks_dim -> 0 recovers the totally irreversible wave and its peak shift.")
print()
print("ALL VALIDATION CHECKS PASSED.")

(3) ks->0 cathodic peak  = 0.3500  (analytic 0.4958*sqrt(alpha) = 0.3506, rel err 1.62e-03)
    peak-shift slope = +4.6020 per decade ks_dim (theory +4.6052)
    PASS: ks_dim -> 0 recovers the totally irreversible wave and its peak shift.

ALL VALIDATION CHECKS PASSED.


### Why these checks, and their limits

All three anchors are computed **independently** of the FD voltammetry code: the
Cottrell flux is a closed form; $0.4463$ and $0.4958\sqrt{\alpha}$ are the
Nicholson–Shain coefficients (Bard & Faulkner 2nd ed., §6.2–6.3); the
reversible cross-check re-simulates the same physics with a *different* boundary
condition (Nernstian Dirichlet rather than Butler–Volmer). Agreement of the
kinetic solver with all three — and with the analytically known
$-\ln 10/\alpha$ peak-shift slope — is therefore a genuine cross-validation, not
a self-fulfilling comparison to the book's output.

One honest caveat: the dimensionless peak current converges to its analytic
value to better than $1\%$ at the time step used here, but the surface
three-point gradient carries an $O(\Delta X)$ bias that the Cottrell window
quantifies ($\sim 10^{-3}$); tightening `dz` and raising `DM` shrinks it
further, at the cost of runtime.

## Summary

* Finite electron-transfer kinetics enter through the **Butler–Volmer** surface
  boundary condition, which a three-point elimination turns into an explicit
  formula for the surface concentration; as the dimensionless rate constant
  $k_s^{\dim}=k_s/\sqrt{fvD}\to\infty$ it collapses to the Chapter 5 Nernstian
  condition, so one implicit solver spans the whole reversibility range.
* Slower kinetics **broaden the wave, lower the peak current, and push the peak
  potential negative**. Quantitatively the cathodic peak runs from the
  reversible $0.4463$ down to the irreversible plateau $0.4958\sqrt{\alpha}$,
  and the peak potential shifts by $-\ln 10/\alpha$ per decade of $k_s^{\dim}$
  ($\approx 29.6/(\alpha n)$ mV per decade of scan rate) — the Nicholson–Shain
  diagnostics, reproduced here to better than $1\%$.
* The same Butler–Volmer machinery carries forward to the AC and pulse methods
  of Chapters 7–8, where the kinetic distortion of the wave becomes a tool for
  *measuring* $k_s$ and $\alpha$ rather than merely a complication.
